In [12]:
import random

In [36]:

class TSP:
    def __init__(self, population_size, crossover_probability, mutation_probability, num_of_generations):
        self.cities = ["Benha", "Shebin Al-Kom", "Al-Zagazig", "Al-Mansoura"]
        self.distances = [
            [0, 25, 42, 79],
            [25, 0, 59, 91],
            [42, 59, 0, 63],
            [79, 91, 63, 0]
        ]
        self.__population_size = population_size
        self.__crossover_probability = crossover_probability
        self.__mutation_probability = mutation_probability
        self.__num_of_generations = num_of_generations
        
        self.__population = []
        other_cities = [1, 2, 3]
        for _ in range(self.__population_size):
            path = [0] + random.sample(other_cities, 3) + [0]
            self.__population.append(path)

    def __calculate_fitness(self):
        fitness_values = []
        for path in self.__population:
            total_distance = sum(self.distances[path[i]][path[i+1]] for i in range(len(path)-1))
            fitness = -total_distance
            fitness_values.append(fitness)
        return fitness_values

    def __selection(self, fitness_vals):
        new_population = []
        for _ in range(self.__population_size):
            idx1, idx2 = random.sample(range(self.__population_size), 2)
            winner = self.__population[idx1] if fitness_vals[idx1] >= fitness_vals[idx2] else self.__population[idx2]
            new_population.append(winner.copy())
        return new_population

    def __repair(self, path):
        middle = path[1:-1]
        seen = set()
        repaired = []
        for city in middle:
            if city not in seen and city != 0:
                seen.add(city)
                repaired.append(city)
        missing = [c for c in [1,2,3] if c not in seen]
        random.shuffle(missing)
        repaired.extend(missing)
        return [0] + repaired[:3] + [0]

    def __crossover(self, parent1, parent2, probability):
        if random.random() >= probability:
            return parent1.copy(), parent2.copy()
        
        middle_point = random.randint(1, 3)
        offspring1 = parent1[:middle_point] + parent2[middle_point:]
        offspring2 = parent2[:middle_point] + parent1[middle_point:]
        
        offspring1 = self.__repair(offspring1)
        offspring2 = self.__repair(offspring2)
        
        return offspring1, offspring2

    def __mutation(self, individual, probability):
        ind = individual.copy()
        if random.random() < probability:
            random_index = random.randint(1, 3)
            ind[random_index] = random.randint(1, 3)
            
            ind = self.__repair(ind)
        
        return ind

    def __crossover_mutation(self):
        new_population = []
        for i in range(0, self.__population_size, 2):
            p1 = self.__population[i]
            p2 = self.__population[(i + 1) % self.__population_size]
            child1, child2 = self.__crossover(p1, p2, self.__crossover_probability)
            child1 = self.__mutation(child1, self.__mutation_probability)
            child2 = self.__mutation(child2, self.__mutation_probability)
            new_population.extend([child1, child2])
        self.__population = new_population[:self.__population_size]

    def solution(self):
        best_fitness = float('-inf')
        best_path = None
        for gen in range(self.__num_of_generations):
            fitness_values = self.__calculate_fitness()
            max_idx = fitness_values.index(max(fitness_values))
            current_best = fitness_values[max_idx]
            if current_best > best_fitness:
                best_fitness = current_best
                best_path = self.__population[max_idx]
                distance = -best_fitness
                print(f"\rGen {gen:5d} | Best distance: {distance:4.0f} km", end="")
            self.__population = self.__selection(fitness_values)
            self.__crossover_mutation()
        print()
        return best_path, -best_fitness

    def display_path(self, path, distance):
        names = [self.cities[i] for i in path]
        print("\nBest Route Found:")
        print(" → ".join(names))
        print(f"Total Distance: {distance} KM")


# Run
problem = TSP(50, 0.82, 0.30, 30000)
best_path, best_distance = problem.solution()
problem.display_path(best_path, best_distance)

Gen     0 | Best distance:  221 km

Best Route Found:
Benha → Al-Zagazig → Al-Mansoura → Shebin Al-Kom → Benha
Total Distance: 221 KM
